# Intel Image Classification using CNN

**Name:** Aditya Roshan S (Aadi)
**Student ID:** 24BAI10227
**Program:** B.Tech AI/ML Engineering, VIT Bhopal University
**Component:** 1A - CNN Algorithm Notebook

**Dataset:** Intel Image Classification (Kaggle) — https://www.kaggle.com/datasets/puneet6060/intel-image-classification

This notebook builds, trains, and evaluates a Convolutional Neural Network (CNN) to classify
natural scene images into 6 categories: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`.

## 1. Dataset Description

The Intel Image Classification dataset contains ~25,000 images (150x150 px) of natural scenes
around the world, split into 6 classes.

- `seg_train/` – ~14,000 training images
- `seg_test/` – ~3,000 test images
- `seg_pred/` – ~7,000 unlabeled prediction images (not used here)

Download the dataset from Kaggle and extract it so the folder structure looks like this
(relative to this notebook):

```
data/
├── seg_train/
│   └── seg_train/
│       ├── buildings/
│       ├── forest/
│       ├── glacier/
│       ├── mountain/
│       ├── sea/
│       └── street/
└── seg_test/
    └── seg_test/
        ├── buildings/
        ├── forest/
        ├── glacier/
        ├── mountain/
        ├── sea/
        └── street/
```

In [ ]:
# 1.1 Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow version:", tf.__version__)

In [ ]:
# 1.2 Paths and parameters
TRAIN_DIR = "data/seg_train/seg_train"
TEST_DIR  = "data/seg_test/seg_test"

IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
EPOCHS     = 15
SEED       = 42

## 2. Loading the Data

In [ ]:
# Loading train data and splitting a validation set (80/20 split)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
# 2.1 Quick look at a few training images
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Performance tuning (prefetch + cache) and pixel normalization
AUTOTUNE = tf.data.AUTOTUNE

normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)

## 3. Data Augmentation

Since the dataset is moderate in size, a small augmentation pipeline is added to reduce
overfitting (random flips, rotation, zoom).

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## 4. CNN Model Architecture

A simple 4-block CNN is used:
Conv2D -> MaxPooling -> Conv2D -> MaxPooling -> Conv2D -> MaxPooling -> Conv2D -> MaxPooling
-> Flatten -> Dense -> Dropout -> Dense(6, softmax)

In [ ]:
num_classes = len(class_names)

model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

## 5. Compile and Train

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_intel_cnn.keras', monitor='val_accuracy', save_best_only=True)
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 6. Training Logs — Accuracy & Loss Plots

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Val Accuracy')
plt.legend(loc='lower right')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 7. Evaluation on the Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

In [ ]:
# 7.1 Confusion Matrix and Classification Report
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Intel Image Classification')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# 7.2 Visualize a few predictions on test images
plt.figure(figsize=(12, 10))
for images, labels in test_ds.take(1):
    preds = model.predict(images)
    pred_labels = np.argmax(preds, axis=1)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        true_label = class_names[labels[i]]
        pred_label = class_names[pred_labels[i]]
        color = 'green' if true_label == pred_label else 'red'
        plt.title(f"True: {true_label}\nPred: {pred_label}", color=color)
        plt.axis("off")
plt.suptitle("Sample Test Predictions")
plt.tight_layout()
plt.show()

## 8. Saving the Trained Model\n\nThe saved model is loaded later inside the Streamlit app (`streamlit_app.py`).

In [ ]:
model.save("intel_cnn_model.keras")
print("Model saved as intel_cnn_model.keras")

## 9. Conclusion

A CNN was trained from scratch on the Intel Image Classification dataset to classify natural
scene images into 6 categories. Data augmentation and early stopping were used to reduce
overfitting. The model's accuracy/loss curves and confusion matrix are shown above, and the
trained model is saved for use in the Streamlit deployment app (Component 1B).

**Possible improvements:** transfer learning (e.g. MobileNetV2 / ResNet50), more aggressive
augmentation, hyperparameter tuning, and a larger training budget.